In [67]:
from pathlib import Path
from datetime import date

import pandas as pd
import numpy as np
from numpy.ma.core import diag

from maps import zsdkap_new_columns_names, zsdkap_dtypes, vbap_new_columns_names, mb5td_new_columns_names, mb5td_dtypes, zek103_new_columns_names, zek103_dtypes, mb52_dtypes, mb52_new_columns_names, cohv_dtypes, cohv_new_columns_names
from py_rfc_methods import get_delivery_plants_and_special_stock_indicators_df
from helper_functions import clean_number

In [68]:
SAP_SYSTEM = "P11_SSO"
STORAGE_LOCATIONS = ['0004', '0005']
SUPPLYING_PLANT = '2101'

In [69]:
source_files_dir = Path(r"P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\18_MISSING_STOCK_ORDERS_IDENTIFIER")
output_files_dir = Path(r"P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\18_MISSING_STOCK_ORDERS_IDENTIFIER\output")

# 1. Define filenames in ONE place using a dictionary
source_file_names = {
    "zsdkap": "ZSDKAP.csv",
    "mb5td": "MB5TD.XLSX",
    "zek103": "ZEK103.XLSX",
    "mb52": "MB52.XLSX",
    "cohv": "COHV.XLSX"
}

output_file_names = {
    "df_zar": "df_ZAR.xlsx",
    "df_zri": "df_ZRI.xlsx",
    "df_zrv": "df_ZRV.xlsx",
}

source_files = {key: source_files_dir / name for key, name in source_file_names.items()}
output_files = {key: output_files_dir / name for key, name in output_file_names.items()}

In [70]:
# product_names = ('ZAR')
# mrp_controllers = ('L2B', 'L2R', 'LI2', 'LI4', 'LI7')

# product_names = ('ZRE_M', 'ZRE M', 'ZRV_M', 'ZRV M')
# mrp_controllers = ('L2E', 'L2V', 'LI1', 'LI3')
#
product_names = ('ZRI', )
mrp_controllers = ('L2I', )

In [71]:
zsdkap = pd.read_csv(source_files["zsdkap"], dtype=zsdkap_dtypes, sep=';', encoding='MacRoman')

In [72]:
df = zsdkap.copy()

In [73]:
df = df.rename(columns=zsdkap_new_columns_names)
df = df[(df['mrp_controller'].isin(mrp_controllers)) & (df['mat_description'].str.startswith(product_names))].reset_index()

In [74]:
df['orders_quantity'] = df['orders_quantity'].apply(clean_number)
df['orders_quantity'] = df['orders_quantity'].astype(int)
df['customer_order_position'] = df['customer_order_position'].str.zfill(6)
df['dispatch_date'] = pd.to_datetime(df['dispatch_date'], dayfirst=True, errors='coerce')
df['creation_date'] = pd.to_datetime(df['creation_date'], dayfirst=True, errors='coerce')

In [75]:
columns_to_drop = ['Land', 'Postleitzahl', 'Ort', 'Strasse', 'Name', 'UPS', 'Materialgruppe', 'Materialgruppenbezeichnung', 'Bestellnummer', 'Versandbedingung', 'Uhrzeiten', 'Kopfnotiz 3', 'Kopfnotiz 4', 'Gewicht', 'Volumen', 'Lieferpriorität', 'Route', 'Transportzone WE', 'Wskaźnik przetw. specj.', 'ID kontenera', 'Spedition']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

In [76]:
delivery_plants = get_delivery_plants_and_special_stock_indicators_df(SAP_SYSTEM, df['customer_order_number'].tolist(), 1000, 100)
delivery_plants = delivery_plants.rename(columns=vbap_new_columns_names)

2026-07-13 16:33:23 | INFO     | SAP_CONN | Buduję parametry dla systemu P11_SSO: {'mshost': 'rffsp11s.sap.roto-frank.com', 'msserv': 'sapmsP11', 'sysid': 'P11', 'group': 'ROTO_FRANK', 'client': '151', 'lang': 'PL', 'snc_mode': '1', 'snc_qop': '8', 'snc_partnername': ' ', 'timeout': '300'}
2026-07-13 16:33:23 | INFO     | SAP_CONN | Połączenie z SAP nawiązane (system=P11_SSO).
2026-07-13 16:33:23 | INFO     | SAP_CONN | Połączenie SAP zamknięte (system=P11_SSO).


In [77]:
df = pd.merge(df, delivery_plants, how='left', on=['customer_order_number', 'customer_order_position'])
df.dropna(subset=['delivery_plant'], inplace=True)

In [78]:
df = df[df['special_stock_indicator'] != 'E']

In [79]:
# Newest dispatch_date first, oldest creation_date second
df_sorted = df.sort_values(by=['dispatch_date', 'creation_date'], ascending=[False, True])

In [80]:
df_grouped = df.groupby(['delivery_plant', 'mat_number', 'dispatch_date'], as_index=False).agg({
    'orders_quantity': 'sum',
    'mat_description': 'first',
})

In [81]:
df_grouped.head(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description
0,0301,491009,2026-07-17,1,ZRI M 054/098 84R8 WIS1B
1,0301,491011,2026-07-14,1,ZRI M 054/118 84R8 WIS1B
2,0301,491013,2026-07-16,1,ZRI M 065/098 84R8 WIS1B
3,0301,491013,2026-07-20,1,ZRI M 065/098 84R8 WIS1B
4,0301,491015,2026-07-14,1,ZRI M 065/118 84R8 WIS1B


In [82]:
sap_list = df['mat_number'].tolist()

In [83]:
zek103_df = pd.read_excel(source_files['zek103'], dtype=zek103_dtypes)
zek103_df.rename(columns=zek103_new_columns_names, inplace=True)

In [84]:
zek103_df['po_delivery_date'] = pd.to_datetime(zek103_df['po_delivery_date'], dayfirst=True, errors='coerce')
zek103_df['po_dispatch_date'] = pd.to_datetime(zek103_df['po_dispatch_date'], dayfirst=True, errors='coerce')
zek103_df = zek103_df[zek103_df['mat_number'].isin(sap_list)]

In [85]:
cols_to_drop = ['Szuk. ciąg', 'LFT', 'Best-Mg', 'Bestät. Menge', 'ME', 'Best. Liefdat.']
zek103_df = zek103_df.drop(columns=[col for col in cols_to_drop if col in zek103_df.columns])

In [86]:
zek103_df.head(5)

,purchase_order_number,purchase_order_position,mat_number,mat_description_zek103,po_delivery_date,po_quantity,plant,po_dispatch_date,customer_order_number,customer_order_position
114,4500133830,60,491030,ZRI M 094/140 84R8 AIS1B,2026-07-15,2.0,0301,2026-07-13,<NA>,0
117,4500134334,40,491041,ZRI M 134/140 84R8 WIS1B,2026-07-17,5.0,0301,2026-07-15,<NA>,0
128,4500135315,50,628092,ZRI M 134/140 84R8 WIT1B,2026-07-15,5.0,0301,2026-07-13,<NA>,0
136,4500135634,10,628088,ZRI M 114/140 84R8 WIT1B,2026-07-17,5.0,0301,2026-07-15,<NA>,0
158,4500136400,60,491033,ZRI M 114/118 84R8 WIS1B,2026-07-14,5.0,0301,2026-07-10,<NA>,0


In [87]:
# Drop purchase orders linked to special customer requirements
zek103_df = zek103_df[zek103_df['customer_order_number'].isna()]

In [88]:
zek103_2_df = zek103_df.copy() # I copy ZEK103 df to get quantities which will be dispatched from production plant

In [89]:
zek103_df = zek103_df.groupby(['po_delivery_date', 'plant', 'mat_number'], as_index=False).agg({
    'po_quantity': 'sum',
    'po_dispatch_date': 'first',
    'mat_description_zek103': 'first',

})

In [90]:
zsdkap_zek103_merged_df = pd.merge(df_grouped, zek103_df, how='outer', left_on=['delivery_plant', 'mat_number', 'dispatch_date'], right_on=['plant', 'mat_number', 'po_delivery_date'])

In [91]:
zsdkap_zek103_merged_df['delivery_plant'] = zsdkap_zek103_merged_df['delivery_plant'].combine_first(zsdkap_zek103_merged_df['plant'])
zsdkap_zek103_merged_df['dispatch_date'] = zsdkap_zek103_merged_df['dispatch_date'].combine_first(zsdkap_zek103_merged_df['po_delivery_date'])
zsdkap_zek103_merged_df['mat_description'] = zsdkap_zek103_merged_df['mat_description'].combine_first(zsdkap_zek103_merged_df['mat_description_zek103'])

In [92]:
zsdkap_zek103_merged_df['orders_quantity'] = zsdkap_zek103_merged_df['orders_quantity'].fillna(0)
zsdkap_zek103_merged_df['po_quantity'] = zsdkap_zek103_merged_df['po_quantity'].fillna(0)

In [93]:
zsdkap_zek103_merged_df = zsdkap_zek103_merged_df.drop(columns=['plant', 'po_delivery_date', 'mat_description_zek103'])

In [94]:
zsdkap_zek103_merged_df.head(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date
0,0301,491009,2026-07-17,1.0,ZRI M 054/098 84R8 WIS1B,0.0,NaT
1,0301,491011,2026-07-14,1.0,ZRI M 054/118 84R8 WIS1B,0.0,NaT
2,0301,491013,2026-07-16,1.0,ZRI M 065/098 84R8 WIS1B,0.0,NaT
3,0301,491013,2026-07-20,1.0,ZRI M 065/098 84R8 WIS1B,0.0,NaT
4,0301,491015,2026-07-14,1.0,ZRI M 065/118 84R8 WIS1B,5.0,2026-07-10


In [95]:
mb52_df = pd.read_excel(source_files['mb52'], dtype=mb52_dtypes)
mb52_df.rename(columns=mb52_new_columns_names, inplace=True)
mb52_df = mb52_df[(mb52_df['mat_number'].isin(sap_list)) & (mb52_df['customer_order_number'].isna()) & (mb52_df['storage_location'].isin(STORAGE_LOCATIONS))]

In [96]:
mb52_df = mb52_df.groupby(['plant', 'mat_number'], as_index=False).agg({
    'stock_quantity': 'sum',
})

In [97]:
mb52_df.head()

,plant,mat_number,stock_quantity
0,0301,491009,7.0
1,0301,491011,7.0
2,0301,491013,6.0
3,0301,491015,10.0
4,0301,491017,5.0


In [98]:
zsdkap_zek103_mb52_merged_df = pd.merge(zsdkap_zek103_merged_df, mb52_df, left_on=['delivery_plant', 'mat_number'], right_on=['plant', 'mat_number']).drop(columns=['plant'])

In [99]:
zsdkap_zek103_mb52_merged_df.head(10)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity
0,0301,491009,2026-07-17,1.0,ZRI M 054/098 84R8 WIS1B,0.0,NaT,7.0
1,0301,491011,2026-07-14,1.0,ZRI M 054/118 84R8 WIS1B,0.0,NaT,7.0
2,0301,491013,2026-07-16,1.0,ZRI M 065/098 84R8 WIS1B,0.0,NaT,6.0
3,0301,491013,2026-07-20,1.0,ZRI M 065/098 84R8 WIS1B,0.0,NaT,6.0
4,0301,491015,2026-07-14,1.0,ZRI M 065/118 84R8 WIS1B,5.0,2026-07-10,10.0
5,0301,491015,2026-07-15,0.0,ZRI M 065/118 84R8 WIS1B,5.0,2026-07-13,10.0
6,0301,491015,2026-07-16,1.0,ZRI M 065/118 84R8 WIS1B,0.0,NaT,10.0
7,0301,491015,2026-07-17,1.0,ZRI M 065/118 84R8 WIS1B,0.0,NaT,10.0
8,0301,491015,2026-07-31,2.0,ZRI M 065/118 84R8 WIS1B,0.0,NaT,10.0
9,0301,491017,2026-07-14,0.0,ZRI M 065/140 84R8 WIS1B,4.0,2026-07-10,5.0


In [100]:
cohv_df = pd.read_excel(source_files['cohv'], dtype=cohv_dtypes)
cohv_df = cohv_df.rename(columns=cohv_new_columns_names)
cohv_df['production_date'] = pd.to_datetime(cohv_df['production_date'], dayfirst=True, errors='coerce')

In [101]:
cohv_df = cohv_df[cohv_df['mat_number'].isin(sap_list)]
cohv_df = cohv_df.groupby(['production_plant', 'mat_number', 'production_date'], as_index=False).agg({
    'production_quantity': 'sum',
})

In [102]:
cohv_df.head(5)

,production_plant,mat_number,production_date,production_quantity
0,2101,2027472,2026-07-14,5
1,2101,2027472,2026-07-15,5
2,2101,491017,2026-07-16,4
3,2101,491027,2026-07-17,5
4,2101,491030,2026-07-16,2


In [103]:
# TODO: Here I lost production data
zsdkap_zek103_mb52_cohv_merged_df = pd.merge(zsdkap_zek103_mb52_merged_df, cohv_df, left_on=['delivery_plant', 'mat_number', 'dispatch_date'], right_on=['production_plant', 'mat_number', 'production_date'], how='left').drop(columns=['production_date', 'production_plant'])

In [104]:
zsdkap_zek103_mb52_cohv_merged_df['production_quantity'] = zsdkap_zek103_mb52_cohv_merged_df['production_quantity'].fillna(0)

In [105]:
zsdkap_zek103_mb52_cohv_merged_df[zsdkap_zek103_mb52_cohv_merged_df['delivery_plant'] == '2101'].head(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity,production_quantity
83,2101,2027472,2026-07-15,12.0,ZRI M 074/xxx x___ WIS1W,0.0,NaT,12.0,5.0


In [106]:
mb5td_df = pd.read_excel(source_files["mb5td"], dtype=mb5td_dtypes)
mb5td_df.rename(columns=mb5td_new_columns_names, inplace=True)
mb5td_df = mb5td_df[mb5td_df['mat_number'].isin(sap_list)]

In [107]:
mb5td_df = mb5td_df[mb5td_df['special_stock_indicator'].isna()]

In [108]:
mb5td_df = mb5td_df[['mat_number', 'supplying_plant', 'purchase_order_number', 'purchase_order_position', 'transit_quantity']]

In [109]:
mb5td_df.head()

,mat_number,supplying_plant,purchase_order_number,purchase_order_position,transit_quantity
102,491033,2101,4500136400,60,5.0
343,491015,2101,4500137915,50,5.0
347,491017,2101,4500137918,60,4.0
439,491015,2101,4500138218,10,5.0
450,628081,2101,4500138221,60,4.0


In [110]:
zek103_2_df.head(5)

,purchase_order_number,purchase_order_position,mat_number,mat_description_zek103,po_delivery_date,po_quantity,plant,po_dispatch_date,customer_order_number,customer_order_position
114,4500133830,60,491030,ZRI M 094/140 84R8 AIS1B,2026-07-15,2.0,0301,2026-07-13,<NA>,0
117,4500134334,40,491041,ZRI M 134/140 84R8 WIS1B,2026-07-17,5.0,0301,2026-07-15,<NA>,0
128,4500135315,50,628092,ZRI M 134/140 84R8 WIT1B,2026-07-15,5.0,0301,2026-07-13,<NA>,0
136,4500135634,10,628088,ZRI M 114/140 84R8 WIT1B,2026-07-17,5.0,0301,2026-07-15,<NA>,0
158,4500136400,60,491033,ZRI M 114/118 84R8 WIS1B,2026-07-14,5.0,0301,2026-07-10,<NA>,0


In [111]:
zek103_2_mb5td_merged_df = pd.merge(zek103_2_df, mb5td_df, on=['mat_number', 'purchase_order_number', 'purchase_order_position'], how='left')

In [112]:
zek103_2_mb5td_merged_df['transit_quantity'] = zek103_2_mb5td_merged_df['transit_quantity'].fillna(0)
zek103_2_mb5td_merged_df['supplying_plant'] = zek103_2_mb5td_merged_df['supplying_plant'].fillna(SUPPLYING_PLANT)
zek103_2_mb5td_merged_df['dispatched_quantity'] = zek103_2_mb5td_merged_df['po_quantity'] - zek103_2_mb5td_merged_df['transit_quantity']

In [113]:
zek103_2_mb5td_merged_df.head(10)

,purchase_order_number,purchase_order_position,mat_number,mat_description_zek103,po_delivery_date,po_quantity,plant,po_dispatch_date,customer_order_number,customer_order_position,supplying_plant,transit_quantity,dispatched_quantity
0,4500133830,60,491030,ZRI M 094/140 84R8 AIS1B,2026-07-15,2.0,0301,2026-07-13,<NA>,0,2101,0.0,2.0
1,4500134334,40,491041,ZRI M 134/140 84R8 WIS1B,2026-07-17,5.0,0301,2026-07-15,<NA>,0,2101,0.0,5.0
2,4500135315,50,628092,ZRI M 134/140 84R8 WIT1B,2026-07-15,5.0,0301,2026-07-13,<NA>,0,2101,0.0,5.0
3,4500135634,10,628088,ZRI M 114/140 84R8 WIT1B,2026-07-17,5.0,0301,2026-07-15,<NA>,0,2101,0.0,5.0
4,4500136400,60,491033,ZRI M 114/118 84R8 WIS1B,2026-07-14,5.0,0301,2026-07-10,<NA>,0,2101,5.0,0.0
5,4500137034,50,628075,ZRI M 074/118 84R8 WIT1B,2026-07-20,5.0,0301,2026-07-16,<NA>,0,2101,0.0,5.0
6,4500137335,30,628082,ZRI M 094/140 84R8 WIT1B,2026-07-21,5.0,0301,2026-07-17,<NA>,0,2101,0.0,5.0
7,4500137915,50,491015,ZRI M 065/118 84R8 WIS1B,2026-07-15,5.0,0301,2026-07-13,<NA>,0,2101,5.0,0.0
8,4500137916,60,491033,ZRI M 114/118 84R8 WIS1B,2026-07-17,5.0,0301,2026-07-15,<NA>,0,2101,0.0,5.0
9,4500137917,20,491041,ZRI M 134/140 84R8 WIS1B,2026-07-15,5.0,0301,2026-07-13,<NA>,0,2101,0.0,5.0


In [114]:
dispatches_df = zek103_2_mb5td_merged_df.groupby(['mat_number', 'supplying_plant', 'po_dispatch_date'], as_index=False).agg({
    'dispatched_quantity': 'sum',
    # 'mat_description_zek103': 'first',
})

In [115]:
dispatches_df = dispatches_df[dispatches_df['dispatched_quantity'] > 0]

In [116]:
dispatches_df.head()

,mat_number,supplying_plant,po_dispatch_date,dispatched_quantity
3,491017,2101,2026-07-16,4.0
4,491027,2101,2026-07-17,5.0
5,491030,2101,2026-07-13,2.0
6,491030,2101,2026-07-16,2.0
8,491033,2101,2026-07-14,5.0


In [117]:
final_df = zsdkap_zek103_mb52_cohv_merged_df.copy()

In [118]:
final_df[final_df['delivery_plant'] == '2101'].head(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity,production_quantity
83,2101,2027472,2026-07-15,12.0,ZRI M 074/xxx x___ WIS1W,0.0,NaT,12.0,5.0


In [119]:
final_df = (pd.merge(final_df, dispatches_df, left_on=['delivery_plant', 'mat_number', 'dispatch_date'], right_on=['supplying_plant', 'mat_number', 'po_dispatch_date'], suffixes=('', '_disp'), how='outer'))
final_df['dispatch_date'] = final_df['dispatch_date'].combine_first(final_df['po_dispatch_date_disp'])
final_df['delivery_plant'] = final_df['delivery_plant'].combine_first(final_df['supplying_plant'])
final_df = final_df.drop(columns=['po_dispatch_date_disp', 'supplying_plant'])

final_df['orders_quantity'] = final_df['orders_quantity'].fillna(0)
final_df['dispatched_quantity'] = final_df['dispatched_quantity'].fillna(0)
final_df['po_quantity'] = final_df['po_quantity'].fillna(0)
final_df['mat_description'] = final_df.groupby('mat_number')['mat_description'].ffill().bfill() # This line fulfills mat description from other rows with the same mat number
final_df['stock_quantity'] = final_df.groupby('mat_number')['stock_quantity'].ffill().bfill()   # This line fulfills stock quantity from other rows with the same mat number

In [120]:
final_df.sample(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity,production_quantity,dispatched_quantity
51,0301,491041,2026-07-15,3.0,ZRI M 134/140 84R8 WIS1B,5.0,2026-07-13,4.0,0.0,0.0
9,0301,491017,2026-07-14,0.0,ZRI M 065/140 84R8 WIS1B,4.0,2026-07-10,5.0,0.0,0.0
20,0301,491029,2026-07-13,1.0,ZRI M 094/140 84R8 WIS1B,0.0,NaT,37.0,0.0,0.0
18,0301,491027,2026-07-21,0.0,ZRI M 094/118 84R8 WIS1B,5.0,2026-07-17,22.0,0.0,0.0
63,0301,628075,2026-07-20,0.0,ZRI M 074/118 84R8 WIT1B,5.0,2026-07-16,9.0,0.0,0.0


In [121]:
# Sort the rows by plant, material number, and date
final_df = final_df.sort_values(by=['delivery_plant', 'mat_number', 'dispatch_date']).reset_index(drop=True)

In [122]:
# Calculate the running totals (cumulative sums) for orders and POs within each group
final_df['cum_orders'] = final_df.groupby(['delivery_plant', 'mat_number'])['orders_quantity'].cumsum()
final_df['cum_po'] = final_df.groupby(['delivery_plant', 'mat_number'])['po_quantity'].cumsum()
final_df['cum_prod'] = final_df.groupby(['delivery_plant', 'mat_number'])['production_quantity'].cumsum()
final_df['cum_dispatches'] = final_df.groupby(['delivery_plant', 'mat_number'])['dispatched_quantity'].cumsum()

# Compute the final stock left column
final_df['stock_left'] = final_df['stock_quantity'] + final_df['cum_po'] - final_df['cum_orders'] + final_df['cum_prod'] - final_df['cum_dispatches']

In [123]:
# 6. Drop the temporary cumulative columns
final_df = final_df.drop(columns=['cum_orders', 'cum_po', 'cum_prod', 'cum_dispatches'])

In [124]:
final_df.head()

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity,production_quantity,dispatched_quantity,stock_left
0,0301,491009,2026-07-17,1.0,ZRI M 054/098 84R8 WIS1B,0.0,NaT,7.0,0.0,0.0,6.0
1,0301,491011,2026-07-14,1.0,ZRI M 054/118 84R8 WIS1B,0.0,NaT,7.0,0.0,0.0,6.0
2,0301,491013,2026-07-16,1.0,ZRI M 065/098 84R8 WIS1B,0.0,NaT,6.0,0.0,0.0,5.0
3,0301,491013,2026-07-20,1.0,ZRI M 065/098 84R8 WIS1B,0.0,NaT,6.0,0.0,0.0,4.0
4,0301,491015,2026-07-14,1.0,ZRI M 065/118 84R8 WIS1B,5.0,2026-07-10,10.0,0.0,0.0,14.0


In [125]:
final_df.to_excel(output_files['df_zri'], index=False)